## Deep Local Volatility with Sobolev Training
In this notebook a neural network is trained to replicate the price of options from a given synthetic volatility surface. The network is trained with and without no arbitrage criteria. The aim is to reconstruct the local volatility function using the Dupire formula and automatic differentiation from the neural network.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import gradcheck
from tqdm.auto import trange
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import seaborn as sns

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and Prepare Dataset

In [ ]:
df = pd.read_csv('LVOptionPrices.csv')

In [ ]:
price_surface = df.pivot(index='K', columns='T', values='Price')
X, Y = np.meshgrid(price_surface.columns, price_surface.index)
Z = price_surface.values
Z.shape
dC_dK, dC_dT = np.gradient(Z)
dC_dT /= 0.15
d2C_dK2, _ = np.gradient(dC_dK)
d2C_dK2 /= (0.05)**2
strikes = df['K'].unique()
strikes_sq = strikes**2
strikes_sq = strikes_sq.reshape(len(strikes), 1)
res = strikes_sq * d2C_dK2
local_volatility_squared = (2 * dC_dT) / (d2C_dK2 * strikes_sq)
local_volatility_squared = np.maximum(local_volatility_squared, 0.0)
local_volatility = np.sqrt(local_volatility_squared)

In [ ]:
def plot_and_save_hm(filename, cmap, vmin=0.3, vmax=0.35, display=True, n = 10):
    fig, ax = plt.subplots(figsize=(14, 10))
    cax = sns.heatmap(vol_values, ax=ax, cmap=cmap, cbar=True, vmin=vmin, vmax=vmax)
    ax.set_xlabel('Stock Price S')
    ax.set_ylabel('Time t')
    
    # Set x-ticks and y-ticks to show only every nth label
    xticks = range(len(Sr))
    yticks = range(len(tr))
    
    # Define which ticks to show labels for
    xticklabels = [label if i % n == 0 else '' for i, label in enumerate(Sr)]
    yticklabels = [label if i % n == 0 else '' for i, label in enumerate(tr)]

    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    if not display:
        plt.close(fig)

In [ ]:
vol_values = local_volatility.T

In [ ]:
Strikes = df['K'].unique()
Maturities = df['T'].unique()
Sr = np.round(Strikes, 2)
tr = np.round(Maturities, 2)

In [ ]:
plot_and_save_hm('lv_mc.png', 'viridis', vmin=0.4, vmax=0.7, display=True, n = 1)

In [ ]:
plot_and_save_hm('lv_mc_gray.png', 'gray', vmin=0.4, vmax=0.7, display=False, n = 1)

In [ ]:
df.drop('ImpliedVol', axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
X_train = df[['K', 'T']]
y_train = df[['Price', 'dC_dK', 'dC_dT']]

In [ ]:
standard_scaler = StandardScaler()
X_train = standard_scaler.fit_transform(X_train)

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train.to_numpy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=len(y_train), 
                                           shuffle=True, drop_last=False)

In [ ]:
def train_model(model, train_loader, loss_fn, optimizer, lambda_1, epochs, scale, scheduler):
    train_errors = []
    test_errors = []
    grad_errors = []
    print(scale)

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        grad_loss_accum = 0.0

        for batch_X, batch_ydy in train_loader:
            batch_X.requires_grad_(True)  # Ensure gradients can be computed for the input

            # Split the input labels into value and gradients
            batch_y = batch_ydy[:, 0]
            dy_dx = batch_ydy[:, 1:]

            # Forward pass
            outputs = model(batch_X)
            value_loss = loss_fn(outputs.squeeze(), batch_y)

            # Compute gradients of outputs with respect to inputs
            outputs_grad = grad(outputs.sum(), batch_X, create_graph=True)[0]
            
            outputs_grad = outputs_grad / scale
            grad_loss = loss_fn(outputs_grad, dy_dx)

            # Combine losses
            loss = value_loss + lambda_1 * grad_loss

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += value_loss.item() * batch_X.size(0)
            grad_loss_accum += grad_loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        grad_loss_accum /= len(train_loader.dataset)
        train_errors.append(train_loss)
        grad_errors.append(grad_loss)
        
        scheduler.step(train_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.9f}, Grad loss: {grad_loss_accum:.9f}")

    history = dict()
    history['train_loss'] = train_errors
    history['grad_loss'] = grad_errors
    return history


### Train  model

In [ ]:
class SoftPlus(nn.Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def forward(self, x):
        return torch.log(1 + torch.exp(x))

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, width = 32, use_batch_norm=False):
        super(NeuralNetwork, self).__init__()
        self.use_batch_norm = use_batch_norm
        self.activation = SoftPlus()
        self.fc1 = nn.Linear(2, width)
        self.bn1 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc5 = nn.Linear(width, 1)

    def forward(self, x):
        x = self.activation(self.bn1(self.fc1(x)))
        x = self.activation(self.bn2(self.fc2(x)))
        x = torch.sigmoid(self.fc5(x))
        return x

In [ ]:
no_epochs = 30000
loss_fn = nn.MSELoss()

In [ ]:
volsob = NeuralNetwork(256, True).to(device)
optimizer = optim.Adam(volsob.parameters(), lr=0.001)
N = 500
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=N, factor=0.3)
scale = torch.tensor(standard_scaler.scale_).float().to(device)

In [ ]:
history = train_model(volsob, train_loader, loss_fn, optimizer, 1.0, no_epochs, scale, scheduler)

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

train_loss_values = history['train_loss']
ax.plot(train_loss_values, color='black', linestyle='-',label = 'training loss', )

ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.set_ylim(0.0, 1.0e-4)
ax.legend()
fig.savefig('VolsobLoss.png', dpi=300, bbox_inches='tight')

In [ ]:
model_path = "volsob.pth"
torch.save(volsob.state_dict(), model_path)

In [ ]:
volsob.eval()

In [ ]:
prices = volsob(train_x).squeeze()

In [ ]:
recondf = pd.DataFrame({
    'Price': prices.detach().cpu().numpy(),
    'K': df['K'].to_numpy().squeeze(),
    'T': df['T'].to_numpy().squeeze()
})

In [ ]:
def plotdfcolumn(filename, dfin, cmap, colname, display=True):
    # Pivot the DataFrame to create a grid of implied volatilities
    vol_surface = dfin.pivot(index='K', columns='T', values='Price')

    # Creating the 3D plot
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Preparing the data for plotting
    X, Y = np.meshgrid(vol_surface.columns, vol_surface.index)
    Z = vol_surface.values

    # Plotting the surface
    surf = ax.plot_surface(X, Y, Z, cmap=cmap, edgecolor='none', antialiased=False)

    # Adding labels and title
    ax.set_xlabel('Maturity T')
    ax.set_ylabel('Strike K')
    ax.set_zlabel(colname)
    ax.set_xlim(2, 0.5)
    ax.view_init(azim=30)

    # Adding a color bar which maps values to colors
    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    if not display:
        plt.close(fig)

In [ ]:
plotdfcolumn('reconstructedpricessob.png', recondf, 'viridis', 'Price')

In [ ]:
plotdfcolumn('reconstructedpricessobgray.png', recondf, 'gray', 'Price', False)

In [ ]:
recondf['Residuals'] = (recondf['Price'] - df['Price'])

In [ ]:
plotdfcolumn('residualssob.png', recondf, 'viridis', 'Residuals')

In [ ]:
plotdfcolumn('residualssobgray.png', recondf, 'gray', 'Residuals', False)

In [ ]:
def prepare_input(K_values, T_values, scaler, device):
    """
    Prepare the input tensor from strike prices and maturities.

    Parameters:
    K_values: Array of strike prices.
    T_values: Array of maturities.
    
    Returns:
    Scaled and flattened tensor for model input.
    """
    # Create a grid of K and T values
    K, T = np.meshgrid(K_values, T_values)
    
    # Combine and scale the inputs
    input_data = np.column_stack((K.ravel(), T.ravel()))

    df = pd.DataFrame(input_data, columns=['K', 'T'])
    input_data_scaled = scaler.transform(df)
    
    # Convert to PyTorch tensor
    input_tensor = torch.tensor(input_data_scaled, dtype=torch.float32).to(device)
    K = torch.tensor(K, dtype=torch.float32).to(device)
    T = torch.tensor(T, dtype=torch.float32).to(device)
    
    return input_tensor, K, T

In [ ]:
Strikes = torch.linspace(0.5, 1.5, 211)  # strike prices
Maturities = torch.linspace(0.5, 2.0, 101)  # maturities

In [ ]:
input_tensor, _, _ = prepare_input(Strikes, Maturities, standard_scaler, device)

In [ ]:
def dupire_local_volatility(model, input_tensor, r, scaler):
    """
    Calculate the Dupire local volatility.

    Parameters:
    model: Trained PyTorch model that predicts option prices.
    input_tensor: Input tensor for the model containing scaled strike and maturity.
    r: Risk-free interest rate.
    scaler: StandardScaler object used for input scaling.

    Returns:
    Local volatility surface as a numpy array.
    """
    input_tensor.requires_grad_(True)

    # Predict option prices using the model
    C = model(input_tensor)

    # Calculate gradients with respect to the input
    dC_dInput = grad(C.sum(), input_tensor, create_graph=True)[0]
    
    # Adjust gradients for the scale of input features
    dC_dK, dC_dT = dC_dInput[:, 0], dC_dInput[:, 1]

    # Calculate the second derivative with respect to K (scaled)
    d2C_dK2 = grad(dC_dK.sum(), input_tensor, create_graph=True)[0][:, 0]
    d2C_dK2 /= scaler.scale_[0]**2

    dC_dK, dC_dT = dC_dK / scaler.scale_[0], dC_dT / scaler.scale_[1]
    
    # Reshape K and T from the input_tensor for correct broadcasting
    K, T = input_tensor[:, 0].detach() * scaler.scale_[0] + scaler.mean_[0], \
            input_tensor[:, 1].detach() * scaler.scale_[1] + scaler.mean_[1]

    # Calculate the Dupire local volatility
    local_volatility_squared = (2 * dC_dT) / (d2C_dK2 * K**2)
    local_volatility = torch.sqrt(torch.abs(local_volatility_squared)).detach().cpu().numpy()

    return local_volatility, dC_dT.detach().cpu().numpy(), \
        dC_dK.detach().cpu().numpy(), \
        d2C_dK2.detach().cpu().numpy(), K.detach().cpu().numpy(), \
        T.detach().cpu().numpy(), C.squeeze().detach().cpu().numpy()

In [ ]:
local_volatility_surface, dC_dT, dC_dK, d2C_dK2, K, T, C = dupire_local_volatility(volsob, 
                                                                                   input_tensor, 
                                                                                   0.0, standard_scaler)

In [ ]:
Sr = np.round(Strikes.numpy(), 2)
tr = np.round(Maturities.numpy(), 2)

In [ ]:
vol_values = local_volatility_surface.reshape(len(Maturities), len(Strikes))

In [ ]:
plot_and_save_hm('volsob_lv.png', 'viridis', vmin=0.4, vmax=0.7, display=True, n=10)

In [ ]:
plot_and_save_hm('volsob_lv_gray.png', 'gray', vmin=0.4, vmax=0.7, display=False, n=10)

#### Residuals

In [ ]:
price_surface = df.pivot(index='K', columns='T', values='Price')
X, Y = np.meshgrid(price_surface.columns, price_surface.index)
Z = price_surface.values
Z.shape
dC_dK, dC_dT = np.gradient(Z)
dC_dT /= 0.15
d2C_dK2, _ = np.gradient(dC_dK)
d2C_dK2 /= (0.05)**2
strikes = df['K'].unique()
strikes_sq = strikes**2
strikes_sq = strikes_sq.reshape(len(strikes), 1)
res = strikes_sq * d2C_dK2
local_volatility_squared = (2 * dC_dT) / (d2C_dK2 * strikes_sq)
local_volatility_squared = np.maximum(local_volatility_squared, 0.0)
local_volatility = np.sqrt(local_volatility_squared)

In [ ]:
vol_values = local_volatility.T
Strikes = df['K'].unique()
Maturities = df['T'].unique()
Sr = np.round(Strikes, 2)
tr = np.round(Maturities, 2)

In [ ]:
input_tensor, _, _ = prepare_input(Strikes, Maturities, standard_scaler, device)

In [ ]:
local_volatility_surface, dC_dT, dC_dK, d2C_dK2, K, T, C = dupire_local_volatility(volsob, input_tensor, 
                                                                                   0.0, standard_scaler)

vol_values_sob = local_volatility_surface.reshape(len(Maturities), len(Strikes))

In [ ]:
vol_values -= vol_values_sob

In [ ]:
plot_and_save_hm('volsob_lv_resid.png', 'viridis', vmin=-0.25, vmax=0.25, display=True, n=1)